# Causal Chamber Experiment — TS-BOSS vs Baselines

Real-world benchmark using the **wind tunnel** dataset from the Causal Chamber.

**Reference**: Gamella et al. (2025), *Causal chambers as a real-world physical testbed for AI methodology*, Nature Machine Intelligence. https://doi.org/10.1038/s42256-024-00964-x  
**Dataset repository**: https://github.com/juangamella/causal-chamber  
**Dataset**: `wt_walks_v1` — 16 independent random walks of wind-tunnel actuators (`actuators_random_walk_1` through `actuators_random_walk_16`).

---

### About the ground-truth graph

`causalchamber.ground_truth.graph('wt', 'standard')` returns the **validated directed static adjacency matrix** (N×N) for the wind-tunnel standard configuration. This graph was validated by Gamella et al. via randomized control experiments (Appendix V of their paper). It encodes which variables causally influence which others, **without temporal lag information**. We therefore evaluate directed static structure recovery after collapsing estimated relations across all lags.

### About `lag_max`

The true causal lag structure of the wind tunnel is **unknown** — this is a real physical system. Following the original benchmark notebook, we set `lag_max=10` as the search hyperparameter for all methods. No method has access to a true `tau_max`.

### Methods compared

We compare **TS-BOSS**, **PCMCI+**, **DYNOTEARS**, and **TS-FGES**. TS-FGES requires Java/JVM and is skipped automatically when it is unavailable.

## 0. Setup

In [1]:
# If not installed yet:
#pip install numpy pandas causalchamber tigramite

In [2]:
import sys, os, time
import numpy as np
import pandas as pd

# Make src/ and utils/ importable regardless of launch cwd
CANDIDATE_ROOTS = [
    os.getcwd(),
    os.path.abspath(os.path.join(os.getcwd(), '..')),
]
REPO_ROOT = next(
    (p for p in CANDIDATE_ROOTS
     if os.path.isdir(os.path.join(p, 'src')) and os.path.isdir(os.path.join(p, 'utils'))),
    CANDIDATE_ROOTS[0],
)
for p in [os.path.join(REPO_ROOT, 'src'), os.path.join(REPO_ROOT, 'utils')]:
    if p not in sys.path:
        sys.path.insert(0, p)

# Causal Chamber
import causalchamber.datasets as cc_datasets
from causalchamber.ground_truth import graph as cc_graph

# Tigramite
import tigramite.data_processing as pp
from tigramite.pcmci import PCMCI
from tigramite.independence_tests.parcorr import ParCorr

# TS-BOSS
from tsboss.ts_boss import TSBOSS

# Other methods (same as experiment_helpers.py)
from dynotears.dynotears import from_pandas_dynamic
from dynotears_to_tigramite import dynotears_to_tigramite_graph
from tsfges import run_tsfges
from metrics import evaluate_graph_complete

DOWNLOAD_DIR = os.path.join(REPO_ROOT, 'data', 'causal_chamber')
os.makedirs(DOWNLOAD_DIR, exist_ok=True)
print('Paths OK. REPO_ROOT =', REPO_ROOT)


Citation
--------

If you use our datasets, simulators or Remote Lab for your work, please consider citing:

﻿@article{gamella2025chamber,
  author={Gamella, Juan L. and Peters, Jonas and B{"u}hlmann, Peter},
  title={Causal chambers as a real-world physical testbed for {AI} methodology},
  journal={Nature Machine Intelligence},
  doi={10.1038/s42256-024-00964-x},
  year={2025},
}


Support & feedback
------------------

If you encounter bugs o have feedback, please write us an email at

  support@causalchamber.ai

or leave an issue at

  https://github.com/juangamella/causal-chamber-package/issues

If you are a Remote Lab subscriber, contact us directly through any of
the provided support channels.


Fetching list of available datasets from
  https://causalchamber.s3.eu-central-1.amazonaws.com/downloadables/directory.yaml ... done.
Paths OK. REPO_ROOT = /Users/irene.castillo/ireneflow/TS-BOSS


## 1. Load ground-truth graph

In [3]:
# 16 sensor variables — same subset as benchmark causal_discovery_time.ipynb
VARIABLES = [
    'hatch', 'load_in', 'load_out', 'pot_1', 'pot_2',
    'current_in', 'current_out',
    'pressure_downwind', 'pressure_upwind',
    'rpm_in', 'rpm_out', 'mic',
    'pressure_intake', 'pressure_ambient',
    'signal_1', 'signal_2'
]
N = len(VARIABLES)

# N x N boolean adjacency — validated by Gamella et al. via randomized control experiments
true_adj = cc_graph('wt', 'standard').loc[VARIABLES, VARIABLES].values.astype(bool)
print('Ground-truth adjacency shape:', true_adj.shape)
print('Number of true edges:', true_adj.sum())

Ground-truth adjacency shape: (16, 16)
Number of true edges: 26


## 2. Load and preprocess data

In [4]:
dataset = cc_datasets.Dataset(name='wt_walks_v1', root=DOWNLOAD_DIR, download=True)
print('Available experiments (first 8):', dataset.available_experiments()[:8])

Dataset wt_walks_v1 found in "/Users/irene.castillo/ireneflow/TS-BOSS/data/causal_chamber/wt_walks_v1".
Available experiments (first 8): ['regime_jumps_single', 'actuators_random_walk_9', 'actuators_random_walk_8', 'loads_hatch_mix_slow_run_2', 'actuators_random_walk_6', 'actuators_random_walk_7', 'loads_hatch_mix_slow_run_3', 'loads_hatch_mix_slow_run_1']


In [5]:
EXPERIMENT = 'actuators_random_walk_1'

df_raw = dataset.get_experiment(EXPERIMENT).as_pandas_dataframe()
df = df_raw[VARIABLES].copy()

# Normalize to [0, 1] — same preprocessing as benchmark notebook
df -= df.min()
df /= df.max()

print(f'Data shape: {df.shape}  (T={df.shape[0]}, N={df.shape[1]})')
df.head()

Data shape: (1016, 16)  (T=1016, N=16)


,hatch,load_in,load_out,pot_1,pot_2,current_in,current_out,pressure_downwind,pressure_upwind,rpm_in,rpm_out,mic,pressure_intake,pressure_ambient,signal_1,signal_2
0,0.832354,0.303030,0.7000,0.802168,0.850205,0.074600,0.172564,0.660675,0.348348,0.132782,0.000000,0.298507,0.012471,0.025031,0.044776,0.816667
1,0.825858,0.303030,0.6875,0.822055,0.862870,0.040261,0.176909,0.630401,0.348348,0.117714,0.061749,0.004071,0.009700,0.062578,0.000000,0.850000
2,0.819053,0.323232,0.6750,0.832517,0.866223,0.063351,0.328988,0.601072,0.340514,0.113188,0.138662,0.002714,0.000000,0.042553,0.000000,0.866667
3,0.818744,0.353535,0.7000,0.833553,0.878935,0.143872,0.440099,0.580574,0.319217,0.128069,0.246903,0.101764,0.012471,0.053817,0.865672,0.866667
4,0.821219,0.333333,0.7250,0.844015,0.868318,0.052694,0.797641,0.540208,0.318727,0.148731,0.318043,0.004071,0.039261,0.121402,0.074627,0.033333


In [6]:
print('Current data shape:', df.shape)

Current data shape: (1016, 16)


## 3. Build shared input (Tigramite dataframe)

Both methods use `lag_max=10` as the **maximum lag search space** — not as a known ground truth.

- **PCMCI+** searches lags 0–10 and prunes via CI tests (p-value < α).  
- **TS-BOSS** unrolls lags 0–10 as candidate parents and prunes via BIC scoring (GST).  

Both methods perform internal lag selection; `lag_max=10` is just the upper bound of the search space, following the original benchmark notebook. Neither method has access to a true τ_max.

In [7]:
LAG_MAX = 10
PCMCI_ALPHA = 0.01  # default

dataframe = pp.DataFrame(
    df.values,
    datatime={0: np.arange(len(df))},
    var_names=VARIABLES
)
print('Tigramite dataframe ready')

Tigramite dataframe ready


## 4. Real-data evaluation protocol

Causal Chamber gives us:

- **time-series observations**, used by every method to search lags $0,\ldots,10$;
- a validated **directed static ground truth** $X \to Y$, which says who causes whom but not the true lag.

Therefore, we cannot evaluate whether an estimated lag is correct. For evaluation only, we project every estimated relation

$$X_i(t-\tau) \to X_j(t) \quad\mapsto\quad X_i \to X_j.$$

The methods still use all lags during discovery; only their final graphs are collapsed for comparison. Repeated discoveries of the same directed relation at different lags count once, and self-links are excluded because Chamber does not label autoregressive relations.

After this necessary projection, we call the repository's unchanged `evaluate_graph_complete` function. We report:

- **static adjacency**: whether the correct pair of variables is connected;
- **static orientation**: whether the direction $X_i \to X_j$ is correct.

We do **not** interpret or report its lagged, autoregressive, or contemporaneous components here. The single tensor slice used below is only a technical container for the static graph; it does not claim that the ground-truth effects occur at lag zero.

In [8]:
def directed_adjacency_to_tigramite(adjacency):
    """Encode a directed static adjacency matrix using the repo's lag-0 mirror convention."""
    adjacency = np.asarray(adjacency, dtype=bool).copy()
    if adjacency.ndim != 2 or adjacency.shape[0] != adjacency.shape[1]:
        raise ValueError('adjacency must be a square N x N matrix')

    np.fill_diagonal(adjacency, False)
    N = adjacency.shape[0]
    graph = np.full((N, N, 1), '', dtype='<U3')

    for i in range(N):
        for j in range(i + 1, N):
            i_to_j = adjacency[i, j]
            j_to_i = adjacency[j, i]

            if i_to_j and j_to_i:
                # A static simple graph cannot encode both directions separately.
                graph[i, j, 0] = 'o-o'
                graph[j, i, 0] = 'o-o'
            elif i_to_j:
                graph[i, j, 0] = '-->'
                graph[j, i, 0] = '<--'
            elif j_to_i:
                graph[j, i, 0] = '-->'
                graph[i, j, 0] = '<--'

    return graph


def _has_arrow_into_right(mark):
    return bool(mark) and mark.endswith('>')


def _has_arrow_into_left(mark):
    return bool(mark) and mark.startswith('<')


def collapse_timeseries_graph_to_static(estimated_graph):
    """Project a Tigramite time-series graph to a directed static graph."""
    estimated_graph = np.asarray(estimated_graph)
    if estimated_graph.ndim != 3 or estimated_graph.shape[0] != estimated_graph.shape[1]:
        raise ValueError('estimated_graph must have shape (N, N, tau_max + 1)')

    N = estimated_graph.shape[0]
    directed = np.zeros((N, N), dtype=bool)
    unoriented = np.zeros((N, N), dtype=bool)

    # At tau=0, decode mirrored Tigramite endpoint marks, including o-> and <-o.
    for i in range(N):
        for j in range(i + 1, N):
            mark_ij = estimated_graph[i, j, 0]
            mark_ji = estimated_graph[j, i, 0]

            if _has_arrow_into_right(mark_ij) or _has_arrow_into_left(mark_ji):
                directed[i, j] = True
            if _has_arrow_into_right(mark_ji) or _has_arrow_into_left(mark_ij):
                directed[j, i] = True
            if (mark_ij or mark_ji) and not (directed[i, j] or directed[j, i]):
                unoriented[i, j] = unoriented[j, i] = True

    # At tau>0, time order determines source (i, past) -> target (j, present).
    if estimated_graph.shape[2] > 1:
        directed |= (estimated_graph[:, :, 1:] != '').any(axis=2)
    np.fill_diagonal(directed, False)

    static_graph = directed_adjacency_to_tigramite(directed)
    for i in range(N):
        for j in range(i + 1, N):
            if unoriented[i, j] or (directed[i, j] and directed[j, i]):
                static_graph[i, j, 0] = 'o-o'
                static_graph[j, i, 0] = 'o-o'

    return static_graph


TRUE_STATIC_GRAPH = directed_adjacency_to_tigramite(true_adj)


def evaluate_chamber_graph(estimated_graph):
    """Apply the repository evaluation protocol after static projection."""
    estimated_static_graph = collapse_timeseries_graph_to_static(estimated_graph)
    if estimated_static_graph.shape != TRUE_STATIC_GRAPH.shape:
        raise ValueError('estimated graph and Chamber ground truth use different variables')
    return evaluate_graph_complete(TRUE_STATIC_GRAPH, estimated_static_graph)


_identity_check = evaluate_graph_complete(TRUE_STATIC_GRAPH, TRUE_STATIC_GRAPH)
for _category in ('adjacency', 'orientation'):
    assert _identity_check[_category]['precision'] == 1.0
    assert _identity_check[_category]['recall'] == 1.0
    assert _identity_check[_category]['f1_score'] == 1.0

## 5. Run all methods

### 5a. TS-BOSS (default: pd=2)

In [9]:
model_tsboss = TSBOSS(lag_max=LAG_MAX, pd=2, rng=np.random.default_rng(42))
t0 = time.time()
model_tsboss.run_tsboss(dataframe, get_mpdag=True, verbose=False)
runtime_tsboss = time.time() - t0
graph_tsboss = model_tsboss.mpdag['graph']
print(f'TS-BOSS done: {runtime_tsboss:.1f}s')

TS-BOSS done: 246.9s


### 5b. PCMCI+ (ParCorr, pc_alpha=0.01)

In [17]:
pcmci = PCMCI(dataframe=dataframe, cond_ind_test=ParCorr(), verbosity=0)
np.random.seed(13)
t0 = time.time()
results_pcmci = pcmci.run_pcmciplus(tau_max=LAG_MAX, pc_alpha=PCMCI_ALPHA)
runtime_pcmci = time.time() - t0
print(f'PCMCI+ done: {runtime_pcmci:.1f}s')

PCMCI+ done: 1.5s


### 5c. DYNOTEARS (default settings)

In [11]:
t0 = time.time()
dynotears_model = from_pandas_dynamic(df, p=LAG_MAX)
runtime_dynotears = time.time() - t0
graph_dynotears, _ = dynotears_to_tigramite_graph(
    structure_model=dynotears_model,
    tau_max=LAG_MAX,
    var_names=VARIABLES,
)
print(f'DYNOTEARS done: {runtime_dynotears:.1f}s')

DYNOTEARS done: 0.1s


### 5d. TS-FGES (default settings)

In [12]:
try:
    t0 = time.time()
    tsfges_out = run_tsfges(
        data=df.values,
        lag_max=LAG_MAX,
        var_names=VARIABLES,
        penalty_discount=1.0,
        replicating=True,
        verbose=False,
    )
    graph_tsfges = tsfges_out['graph']
    runtime_tsfges = time.time() - t0
    print(f'TS-FGES done: {runtime_tsfges:.1f}s')
except Exception as e:
    graph_tsfges = None
    runtime_tsfges = None
    print(f'TS-FGES skipped: {type(e).__name__}: {e}')

TS-FGES done: 3.0s


## 6. Summarize results

In [13]:
methods = [
    (
        'TS-BOSS',
        globals().get('graph_tsboss'),
        globals().get('runtime_tsboss'),
    ),
    (
        'PCMCI+',
        results_pcmci['graph'] if 'results_pcmci' in globals() else None,
        globals().get('runtime_pcmci'),
    ),
    (
        'DYNOTEARS',
        globals().get('graph_dynotears'),
        globals().get('runtime_dynotears'),
    ),
    (
        'TS-FGES',
        globals().get('graph_tsfges'),
        globals().get('runtime_tsfges'),
    ),
]

records = []
for name, graph, runtime in methods:
    if graph is None or runtime is None:
        records.append({'method': name, 'status': 'not run', 'runtime_s': np.nan})
        continue

    result = evaluate_chamber_graph(graph)
    adjacency = result['adjacency']
    orientation = result['orientation']
    records.append({
        'method': name,
        'status': 'ok',
        'static_adj_precision': adjacency['precision'],
        'static_adj_recall': adjacency['recall'],
        'static_adj_f1': adjacency['f1_score'],
        'static_adj_TP': adjacency['TP'],
        'static_adj_FP': adjacency['FP'],
        'static_adj_FN': adjacency['FN'],
        'static_ori_precision': orientation['precision'],
        'static_ori_recall': orientation['recall'],
        'static_ori_f1': orientation['f1_score'],
        'runtime_s': runtime,
    })

df_results = pd.DataFrame(records).set_index('method')
display(df_results.round(3))

,status,static_adj_precision,static_adj_recall,static_adj_f1,static_adj_TP,static_adj_FP,static_adj_FN,static_ori_precision,static_ori_recall,static_ori_f1,runtime_s
method,,,,,,,,,,,
TS-BOSS,ok,0.306,0.577,0.400,30,68,22,0.119,0.308,0.172,246.920
PCMCI+,ok,0.538,0.269,0.359,14,12,38,0.364,0.154,0.216,1.478
DYNOTEARS,ok,0.400,0.308,0.348,16,24,36,0.161,0.192,0.175,0.142
TS-FGES,ok,0.474,0.346,0.400,18,20,34,0.286,0.308,0.296,3.040


## 7. Aggregate over all actuator random walks

For the paper-level result, we run each method **separately** on all 16 available experiments, `actuators_random_walk_1` through `actuators_random_walk_16`. We do not concatenate trajectories, because doing so would create artificial temporal links between the end of one experiment and the beginning of the next.

Every run uses the same variables, preprocessing, `lag_max`, hyperparameters, and method seeds as the single-walk experiment. Static adjacency and static orientation are evaluated independently for each walk. We report the mean and standard error across walks, $\mathrm{SE}=s/\sqrt{n}$.

The code discovers the actuator random walks from the dataset instead of hard-coding their number. It checkpoints per-walk results to `results/causal_chamber_static_per_walk.csv` after every completed method. A complete run should take approximately 65–80 minutes, mostly due to TS-BOSS.

In [16]:
import contextlib
import gc
import io

EXPERIMENTS = sorted(
    (name for name in dataset.available_experiments() if name.startswith('actuators_random_walk_')),
    key=lambda name: int(name.rsplit('_', 1)[1]),
)
print(f'Found {len(EXPERIMENTS)} actuator random walks:', EXPERIMENTS)

RESULTS_PATH = os.path.join(REPO_ROOT, 'results', 'causal_chamber_static_per_walk.csv')
os.makedirs(os.path.dirname(RESULTS_PATH), exist_ok=True)

# Start fresh. If interrupted, change this to True to resume from the checkpoint.
RESUME = False
if RESUME and os.path.exists(RESULTS_PATH):
    multiwalk_records = pd.read_csv(RESULTS_PATH).to_dict('records')
else:
    multiwalk_records = []

completed = {(r['experiment'], r['method']) for r in multiwalk_records}


def append_static_result(experiment, method, graph, runtime):
    result = evaluate_chamber_graph(graph)
    adjacency = result['adjacency']
    orientation = result['orientation']
    multiwalk_records.append({
        'experiment': experiment,
        'method': method,
        'static_adj_precision': adjacency['precision'],
        'static_adj_recall': adjacency['recall'],
        'static_adj_f1': adjacency['f1_score'],
        'static_ori_precision': orientation['precision'],
        'static_ori_recall': orientation['recall'],
        'static_ori_f1': orientation['f1_score'],
        'runtime_s': runtime,
    })
    pd.DataFrame(multiwalk_records).to_csv(RESULTS_PATH, index=False)


for experiment in EXPERIMENTS:
    print(f'\n{experiment}')
    df_e = dataset.get_experiment(experiment).as_pandas_dataframe()[VARIABLES].copy()
    df_e -= df_e.min()
    df_e /= df_e.max().replace(0, 1)
    dfr = pp.DataFrame(
        df_e.values,
        datatime={0: np.arange(len(df_e))},
        var_names=VARIABLES,
    )

    if (experiment, 'TS-BOSS') not in completed:
        model = TSBOSS(lag_max=LAG_MAX, pd=2, rng=np.random.default_rng(42))
        t0 = time.time()
        # TS-BOSS can print substantial internal progress; retain only the concise status below.
        with contextlib.redirect_stdout(io.StringIO()):
            model.run_tsboss(dfr, get_mpdag=True, verbose=False)
        runtime = time.time() - t0
        append_static_result(experiment, 'TS-BOSS', model.mpdag['graph'], runtime)
        print(f'  TS-BOSS: {runtime:.1f}s')
        del model
        gc.collect()

    if (experiment, 'PCMCI+') not in completed:
        np.random.seed(13)
        pcmci_e = PCMCI(dataframe=dfr, cond_ind_test=ParCorr(), verbosity=0)
        t0 = time.time()
        pcmci_out = pcmci_e.run_pcmciplus(tau_max=LAG_MAX, pc_alpha=PCMCI_ALPHA)
        runtime = time.time() - t0
        append_static_result(experiment, 'PCMCI+', pcmci_out['graph'], runtime)
        print(f'  PCMCI+: {runtime:.1f}s')

    if (experiment, 'DYNOTEARS') not in completed:
        t0 = time.time()
        dynotears_e = from_pandas_dynamic(df_e, p=LAG_MAX)
        runtime = time.time() - t0
        graph_e, _ = dynotears_to_tigramite_graph(
            structure_model=dynotears_e,
            tau_max=LAG_MAX,
            var_names=VARIABLES,
        )
        append_static_result(experiment, 'DYNOTEARS', graph_e, runtime)
        print(f'  DYNOTEARS: {runtime:.1f}s')

    if (experiment, 'TS-FGES') not in completed:
        try:
            t0 = time.time()
            tsfges_e = run_tsfges(
                data=df_e.values,
                lag_max=LAG_MAX,
                var_names=VARIABLES,
                penalty_discount=1.0,
                replicating=True,
                verbose=False,
            )
            runtime = time.time() - t0
            append_static_result(experiment, 'TS-FGES', tsfges_e['graph'], runtime)
            print(f'  TS-FGES: {runtime:.1f}s')
        except Exception as exc:
            print(f'  TS-FGES skipped: {type(exc).__name__}: {exc}')

multiwalk_results = pd.DataFrame(multiwalk_records)
metric_columns = [
    'static_adj_precision', 'static_adj_recall', 'static_adj_f1',
    'static_ori_precision', 'static_ori_recall', 'static_ori_f1',
    'runtime_s',
]

summary_records = []
for method, group in multiwalk_results.groupby('method'):
    row = {'method': method, 'n_walks': len(group)}
    for metric in metric_columns:
        row[f'{metric}_mean'] = group[metric].mean()
        row[f'{metric}_se'] = group[metric].sem()
    summary_records.append(row)

multiwalk_summary = pd.DataFrame(summary_records).set_index('method')
display(multiwalk_summary.round(3))
print('Per-walk checkpoint:', RESULTS_PATH)

Found 16 actuator random walks: ['actuators_random_walk_1', 'actuators_random_walk_2', 'actuators_random_walk_3', 'actuators_random_walk_4', 'actuators_random_walk_5', 'actuators_random_walk_6', 'actuators_random_walk_7', 'actuators_random_walk_8', 'actuators_random_walk_9', 'actuators_random_walk_10', 'actuators_random_walk_11', 'actuators_random_walk_12', 'actuators_random_walk_13', 'actuators_random_walk_14', 'actuators_random_walk_15', 'actuators_random_walk_16']

actuators_random_walk_1
  TS-BOSS: 270.8s
  PCMCI+: 1.5s
  DYNOTEARS: 0.1s
  TS-FGES: 2.3s

actuators_random_walk_2
  TS-BOSS: 368.1s
  PCMCI+: 42.0s
  DYNOTEARS: 0.4s
  TS-FGES: 4.5s

actuators_random_walk_3
  TS-BOSS: 400.7s
  PCMCI+: 51.0s
  DYNOTEARS: 1.2s
  TS-FGES: 4.4s

actuators_random_walk_4
  TS-BOSS: 432.7s
  PCMCI+: 132.0s
  DYNOTEARS: 0.5s
  TS-FGES: 4.7s

actuators_random_walk_5
  TS-BOSS: 415.7s
  PCMCI+: 41.1s
  DYNOTEARS: 1.3s
  TS-FGES: 4.5s

actuators_random_walk_6
  TS-BOSS: 472.3s
  PCMCI+: 33.8s
  DY

,n_walks,static_adj_precision_mean,static_adj_precision_se,static_adj_recall_mean,static_adj_recall_se,static_adj_f1_mean,static_adj_f1_se,static_ori_precision_mean,static_ori_precision_se,static_ori_recall_mean,static_ori_recall_se,static_ori_f1_mean,static_ori_f1_se,runtime_s_mean,runtime_s_se
method,,,,,,,,,,,,,,,
DYNOTEARS,16,0.253,0.015,0.113,0.014,0.153,0.014,0.035,0.011,0.031,0.012,0.032,0.011,0.794,0.112
PCMCI+,16,0.318,0.020,0.276,0.016,0.293,0.015,0.315,0.032,0.180,0.018,0.224,0.021,68.981,11.500
TS-BOSS,16,0.311,0.008,0.630,0.016,0.416,0.010,0.216,0.014,0.442,0.020,0.288,0.016,394.438,23.896
TS-FGES,16,0.395,0.021,0.433,0.024,0.410,0.021,0.249,0.016,0.373,0.022,0.296,0.018,4.516,0.159


Per-walk checkpoint: /Users/irene.castillo/ireneflow/TS-BOSS/results/causal_chamber_static_per_walk.csv
